# Beta as a Regression Slope

CAPM (04/05) defines beta as $\operatorname{Cov}(R_i, R_M)/\sigma_M^2$. Estimating it from data is a linear regression of an asset's excess
returns on the market's. The regression gives three numbers at once: the slope (beta), the intercept (alpha) and the
residual volatility (idiosyncratic risk). This notebook derives the least-squares estimates, shows how precise each one
is, and works through the practical problems: beta that changes over time, prices that are not recorded at the same
moment, and the tendency of extreme betas to move toward 1.

## 0. Math link

- [MLE and linear regression](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/05_point_estimation/04_mle_linear_regression.ipynb):
  with normal errors, maximizing the likelihood is the same as minimizing the sum of squared residuals.
- [Covariance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/06_covariance.ipynb)
  and [correlation](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/09_correlation_coefficient.ipynb):
  the slope is $\operatorname{Cov}(x, y)/\operatorname{Var}(x) = \rho\,\sigma_y/\sigma_x$.
- [Partial derivatives](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/02_gradients/01_partial_derivatives_part_1.ipynb):
  setting both partial derivatives of the squared error to zero gives the estimates.
- CAPM, beta and the security market line from [04/05](../04_portfolio_mathematics/05_capm_and_security_market_line.ipynb);
  testing alpha from [06/03](../06_estimation_error/03_is_this_funds_alpha_real.ipynb).

## 1. Introduction

A fund manager reports that the fund beat the market by 2.6 points a year. If the fund has a beta of 1.2, most of that gap is
market exposure, not skill (Example 3). Separating the two requires a regression.

After this notebook you can:

- derive the least-squares slope and intercept, and connect them to covariance, correlation and maximum likelihood;
- split an asset's variance into systematic and idiosyncratic parts and read $R^2$;
- compute standard errors of beta and alpha and explain why beta is precise and alpha is not;
- estimate time-varying beta with a rolling window;
- correct daily betas for non-synchronous prices, and shrink betas toward 1.

## 2. Theory

### The market model

With excess returns $y_t = r_{i,t} - r_f$ for the asset and $x_t = r_{M,t} - r_f$ for the market,

$$
y_t = \alpha + \beta x_t + \varepsilon_t, \qquad E[\varepsilon_t] = 0,\ \operatorname{Var}(\varepsilon_t) = \sigma_\varepsilon^2,\ \varepsilon_t \text{ uncorrelated with } x_t
$$

CAPM says $\alpha = 0$.

### Least squares

Minimize $S(\alpha, \beta) = \sum_t (y_t - \alpha - \beta x_t)^2$. Setting the partial derivatives to zero:

$$
\frac{\partial S}{\partial\alpha} = -2\sum_t (y_t - \alpha - \beta x_t) = 0, \qquad
\frac{\partial S}{\partial\beta} = -2\sum_t x_t(y_t - \alpha - \beta x_t) = 0
$$

The first gives $\hat\alpha = \bar y - \hat\beta\bar x$. Substituting into the second:

$$
\hat\beta = \frac{\sum_t (x_t - \bar x)(y_t - \bar y)}{\sum_t (x_t - \bar x)^2} = \frac{\widehat{\operatorname{Cov}}(x, y)}{\widehat{\operatorname{Var}}(x)} = \hat\rho\,\frac{\hat\sigma_y}{\hat\sigma_x}
$$

If $\varepsilon_t$ is normal, the log-likelihood is $-\frac{n}{2}\ln(2\pi\sigma_\varepsilon^2) - S(\alpha, \beta)/(2\sigma_\varepsilon^2)$, so maximum likelihood gives the same
$\hat\alpha, \hat\beta$.

### Variance decomposition and $R^2$

Since $\varepsilon$ is uncorrelated with $x$:

$$
\sigma_y^2 = \beta^2\sigma_x^2 + \sigma_\varepsilon^2, \qquad R^2 = \frac{\beta^2\sigma_x^2}{\sigma_y^2} = \rho^2
$$

$\beta^2\sigma_x^2$ is **systematic** risk, $\sigma_\varepsilon^2$ **idiosyncratic** risk. $R^2$ is the systematic share. Idiosyncratic risk diversifies
away in a large portfolio; systematic risk does not (04/05).

### Standard errors

Conditional on the market returns, with $n$ observations:

$$
\operatorname{SE}(\hat\beta) = \frac{\sigma_\varepsilon}{\sqrt{\sum_t (x_t - \bar x)^2}} \approx \frac{\sigma_\varepsilon}{\sigma_x\sqrt n}, \qquad
\operatorname{SE}(\hat\alpha) = \sigma_\varepsilon\sqrt{\frac1n + \frac{\bar x^2}{\sum_t (x_t - \bar x)^2}} \approx \frac{\sigma_\varepsilon}{\sqrt n}
$$

with $\sigma_\varepsilon$, $\sigma_x$ per period. Annualizing as in 06/01: $\operatorname{SE}(\hat\alpha_{\text{ann}}) \approx \sigma_{\varepsilon,\text{ann}}/\sqrt T$, which depends only on the
span; $\operatorname{SE}(\hat\beta) \approx \sigma_{\varepsilon,\text{ann}}/(\sigma_{x,\text{ann}}\sqrt{n})$ shrinks with the number of observations, like a volatility. **Beta is a
second-moment quantity and is estimated precisely; alpha is a mean and is not** (06/04).

### Alpha vs active return

The active return against the market is $y_t - x_t = \alpha + (\beta - 1)x_t + \varepsilon_t$. Its mean is $\alpha + (\beta - 1)E[x]$. A high-beta fund
with no alpha beats the market on average whenever the market premium is positive, and the active-return test of 06/03
would credit it with skill. The regression alpha removes the market exposure.

### Practical problems

- **Time variation.** Betas change with leverage, business mix and the market environment. A rolling window (e.g. 36 or 60
  months) trades precision for responsiveness.
- **Non-synchronous prices.** If a stock's price reacts to market news partly on the next day (illiquid stocks, different
  closing times), same-day daily regressions underestimate beta. Dimson (1979): regress on the current and lagged market
  returns and add the slopes. Monthly data suffers much less.
- **Regression toward 1.** Across many stocks, the highest estimated betas are partly noise and tend to be lower next period
  (Blume 1975), the cross-sectional cousin of 06/03's regression to the mean. A common adjustment is
  $\beta_{\text{adj}} = \tfrac23\hat\beta + \tfrac13$.

## 3. Worked Examples

### Example 1: Swiss equities on global equities

$\rho = 0.8$, $\sigma_{SE} = 16\%$, $\sigma_{GE} = 17\%$: $\beta = 0.8 \cdot 16/17 = 0.753$, $R^2 = 0.64$, idiosyncratic volatility $16\%\sqrt{1 - 0.64} = 9.6\%$.

### Example 2: precision of beta and alpha

Five years of monthly data ($n = 60$), same numbers:
$\operatorname{SE}(\hat\beta) = 9.6\%/(17\%\sqrt{60}) = 0.073$. A 95% interval is $0.753 \pm 0.14$.
$\operatorname{SE}(\hat\alpha_{\text{ann}}) = 9.6\%/\sqrt5 = 4.3\%$ per year. An alpha of 2% would be less than half a standard error.

### Example 3: a high-beta fund

Fund excess return 10.6%, market excess return 8.0% (both annual averages), beta 1.2. Raw outperformance: 2.6 points.
Alpha: $10.6\% - 1.2 \cdot 8.0\% = 1.0\%$. Most of the outperformance was market exposure.

### Example 4: least squares by hand

Five monthly observations (%): $x = (2, -1, 3, 0, 1)$, $y = (3, -2, 4, 1, 1)$. Means $\bar x = 1$, $\bar y = 1.4$.
Deviations $x - \bar x = (1, -2, 2, -1, 0)$, $y - \bar y = (1.6, -3.4, 2.6, -0.4, -0.4)$.
$\sum(x-\bar x)(y-\bar y) = 1.6 + 6.8 + 5.2 + 0.4 + 0 = 14$, $\sum(x-\bar x)^2 = 10$.
$\hat\beta = 1.4$, $\hat\alpha = 1.4 - 1.4 \cdot 1 = 0$.

## 4. Finance Application

### Where it matters

- **Cost of equity.** CAPM cost of capital $r_f + \beta(\mu_M - r_f)$ (04/05) needs a beta; data providers publish regression
  betas, usually from 2–5 years of weekly or monthly data, often Blume-adjusted.
- **Hedging.** To hedge market exposure of a portfolio with index futures, sell $\beta \times$ portfolio value of the index
  (03/03 minimum-variance hedge ratio is the same formula).
- **Performance evaluation.** Fund alpha is the regression intercept against the benchmark, not the raw difference in
  returns; multi-factor versions follow in 07/03.
- **Risk decomposition.** $R^2$ tells how much of a stock's risk is market risk; low-$R^2$ stocks are good diversifiers in a
  portfolio dominated by the market.

### The usual misreadings

- "Beta measures risk." It measures **systematic** risk. A stock with beta 0.5 and 60% volatility is very risky on its
  own; most of its risk is idiosyncratic.
- "The fund beat the index, so it has alpha." Only after removing $(\beta - 1)$ times the market return.
- "A beta of 1.4 estimated from daily data is the stock's beta." For thinly traded stocks daily betas are biased
  toward zero; for stocks in a different time zone from the index as well.
- "Low $R^2$ means the beta estimate is useless." Low $R^2$ makes $\sigma_\varepsilon$ large and the standard error wider, but the
  estimate is still unbiased.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
months_per_year = 12
trading_days = 252

### One stock

Market: global equities with annual excess return 6.5% and volatility 17%. Stock: true beta 1.2, alpha 0, idiosyncratic
volatility 20% per year. Five years of monthly excess returns (decimal, CHF).

In [ ]:
mkt_mu, mkt_vol = 0.065, 0.17          # annual excess return and volatility of the market
beta_true, alpha_true, idio_vol = 1.2, 0.0, 0.20

def simulate_stock(n_obs, rng, beta=beta_true, alpha=alpha_true, idio=idio_vol, ppy=12, n_paths=None):
    shape = (n_obs,) if n_paths is None else (n_paths, n_obs)
    x = mkt_mu / ppy + mkt_vol / np.sqrt(ppy) * rng.standard_normal(shape)
    y = alpha / ppy + beta * x + idio / np.sqrt(ppy) * rng.standard_normal(shape)
    return x, y


def ols(x, y):
    # slope and intercept along the last axis
    xc = x - x.mean(axis=-1, keepdims=True)
    yc = y - y.mean(axis=-1, keepdims=True)
    b = (xc * yc).sum(axis=-1) / (xc**2).sum(axis=-1)
    a = y.mean(axis=-1) - b * x.mean(axis=-1)
    return a, b


x, y = simulate_stock(60, rng)
a_hat, b_hat = ols(x, y)
lr = stats.linregress(x, y)
poly = np.polyfit(x, y, 1)

resid = y - a_hat - b_hat * x
print(f"beta:  ols {b_hat:.4f}, scipy {lr.slope:.4f}, polyfit {poly[0]:.4f};  SE {lr.stderr:.4f}")
print(f"alpha: {a_hat * 12:.2%} per year, SE {lr.intercept_stderr * 12:.2%} per year")
print(f"R^2 {lr.rvalue**2:.3f} (true {beta_true**2 * mkt_vol**2 / (beta_true**2 * mkt_vol**2 + idio_vol**2):.3f}), "
      f"idiosyncratic vol {resid.std(ddof=2) * np.sqrt(12):.2%}")
print(f"cov/var = {np.cov(x, y)[0, 1] / x.var(ddof=1):.4f},  rho * sy/sx = {lr.rvalue * y.std() / x.std():.4f}")

All four routes give the same slope. This history estimates beta at 1.12, half a standard error from the true 1.2, and
alpha at −12% per year, 1.7 standard errors from the true zero: the alpha estimate is far noisier than the beta estimate.

### How precise are beta and alpha?

5,000 histories of five years.

In [ ]:
n_paths = 5_000
X, Y = simulate_stock(60, rng, n_paths=n_paths)
A_hat, B_hat = ols(X, Y)

se_beta = idio_vol / (mkt_vol * np.sqrt(60))
se_alpha = idio_vol / np.sqrt(5)
pd.DataFrame({
    "true": [beta_true, alpha_true],
    "average estimate": [B_hat.mean(), A_hat.mean() * 12],
    "SE simulated": [B_hat.std(), (A_hat * 12).std()],
    "SE formula": [se_beta, se_alpha],
    "SE / |true| (beta)": [B_hat.std() / beta_true, np.nan],
}, index=["beta", "alpha (annual)"]).round(4)

Over five years beta has a standard error of 0.15 (13% of its value); alpha has a standard error of 8.9 points per year
around a true value of zero. Both match the formulas.

### Alpha vs active return for a high-beta fund

1,000 funds with beta 1.2 and **zero** alpha, 10 years each. Test the mean active return against the market (06/03) and
test the regression alpha.

In [ ]:
Xf, Yf = simulate_stock(120, rng, n_paths=1_000)
active = Yf - Xf
t_active = active.mean(axis=1) / (active.std(axis=1, ddof=1) / np.sqrt(120))

Af, Bf = ols(Xf, Yf)
res_f = Yf - Af[:, None] - Bf[:, None] * Xf
s_eps = np.sqrt((res_f**2).sum(axis=1) / (120 - 2))
sxx = ((Xf - Xf.mean(axis=1, keepdims=True)) ** 2).sum(axis=1)
se_a = s_eps * np.sqrt(1 / 120 + Xf.mean(axis=1) ** 2 / sxx)
t_alpha = Af / se_a

print(f"average active return: {active.mean() * 12:.2%} per year  (theory (beta - 1) * 6.5% = {(beta_true - 1) * mkt_mu:.2%})")
print(f"average regression alpha: {Af.mean() * 12:.2%} per year")
print(f"share 'significant' at 5% one-sided: active-return test {np.mean(t_active > 1.645):.1%}, "
      f"regression alpha {np.mean(t_alpha > 1.645):.1%}")

The high-beta fund beats the market by 1.35% per year on average with zero alpha. The active-return test flags 8.2% of
these funds as skilled at the 5% level; the regression alpha test flags 5.2%, the nominal rate. Over 10 years the
market exposure inflates the active-return test only modestly; with a larger beta gap or longer data it would do so more.

### Rolling beta

A stock whose true beta jumps from 0.8 to 1.3 after ten years (e.g. after a leveraged acquisition). Rolling 36- and
60-month betas over 20 years.

In [ ]:
n1 = 120
x_r1, y_r1 = simulate_stock(n1, rng, beta=0.8)
x_r2, y_r2 = simulate_stock(n1, rng, beta=1.3)
x_roll, y_roll = np.concatenate([x_r1, x_r2]), np.concatenate([y_r1, y_r2])
true_path = np.r_[np.full(n1, 0.8), np.full(n1, 1.3)]

def rolling_beta(x, y, window):
    out = np.full(x.size, np.nan)
    for t in range(window, x.size + 1):
        out[t - 1] = ols(x[t - window:t], y[t - window:t])[1]
    return out

roll36, roll60 = rolling_beta(x_roll, y_roll, 36), rolling_beta(x_roll, y_roll, 60)
for m in [119, 131, 155, 179, 239]:
    print(f"month {m + 1:3d}: true {true_path[m]:.1f}, 36m {roll36[m]:.2f}, 60m {roll60[m]:.2f}")

Twelve months after the jump the 36-month beta still reads 0.69 and the 60-month beta 0.99; the 36-month window catches up
after about three years, the 60-month window after five. Both are noisy: at month 240 the 36-month estimate is 1.73 against
a true 1.3.

### Non-synchronous prices and the Dimson correction

Daily data: the stock reacts 60% to today's market return and 40% to yesterday's (true beta 1.2 in total). Compare the
same-day daily regression, the Dimson regression on today's and yesterday's market return, and a monthly regression,
over 1,000 histories of five years.

In [ ]:
def nonsync_estimates(rng, n_years=5, lag_share=0.4):
    n = n_years * trading_days + 1
    xm = mkt_mu / trading_days + mkt_vol / np.sqrt(trading_days) * rng.standard_normal(n)
    ys = (beta_true * ((1 - lag_share) * xm[1:] + lag_share * xm[:-1])
          + idio_vol / np.sqrt(trading_days) * rng.standard_normal(n - 1))
    xs = xm[1:]
    same_day = ols(xs, ys)[1]
    Z = np.column_stack([np.ones(n - 2), xs[1:], xs[:-1]])
    coef = np.linalg.lstsq(Z, ys[1:], rcond=None)[0]
    xm_m = xs[: n_years * trading_days].reshape(-1, 21).sum(axis=1)
    ym_m = ys[: n_years * trading_days].reshape(-1, 21).sum(axis=1)
    return same_day, coef[1] + coef[2], ols(xm_m, ym_m)[1]

ns = np.array([nonsync_estimates(rng) for _ in range(1_000)])
ns_table = pd.DataFrame({"average": ns.mean(axis=0), "std": ns.std(axis=0)},
                        index=["daily, same day", "daily, Dimson (lag 0 + lag 1)", "monthly"])
ns_table.round(3)

The same-day daily beta averages 0.72, the 60% of the true 1.2 that arrives on the same day. The Dimson estimate recovers 1.20.
Monthly data is almost unbiased (1.18: only reactions that spill over a month end are lost) but more than three times as
noisy as the daily estimates.

### Betas regress toward 1

500 stocks with true betas drawn around 1 (standard deviation 0.3). Estimate each beta in two separate five-year periods and
regress the second-period estimates on the first.

In [ ]:
n_stocks = 500
true_betas = 1 + 0.3 * rng.standard_normal(n_stocks)
b1 = np.array([ols(*simulate_stock(60, rng, beta=b))[1] for b in true_betas])
b2 = np.array([ols(*simulate_stock(60, rng, beta=b))[1] for b in true_betas])
slope_12 = stats.linregress(b1, b2).slope
var_true, var_noise = true_betas.var(), (b1 - true_betas).var()

print(f"slope of period-2 betas on period-1 betas: {slope_12:.2f}")
print(f"theory var(true) / (var(true) + var(noise)) = {var_true / (var_true + var_noise):.2f}")
top = b1 >= np.quantile(b1, 0.9)
print(f"top decile by period-1 beta: period 1 {b1[top].mean():.2f}, period 2 {b2[top].mean():.2f}, "
      f"true {true_betas[top].mean():.2f}, Blume-adjusted {(2 / 3 * b1[top] + 1 / 3).mean():.2f}")

The slope of 0.75 is close to the theoretical share of true variation in the estimates, 0.78. The top decile by period-1 beta
averaged 1.57 in period 1 and 1.43 in period 2, close to its true average of 1.44. The Blume adjustment (1.38) moves the
estimate in the right direction, here a little too far.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x, y, s=15, alpha=0.7, label="monthly excess returns")
xx = np.linspace(x.min(), x.max(), 50)
ax.plot(xx, a_hat + b_hat * xx, color="tab:red", label=f"fitted: beta {b_hat:.2f}")
ax.plot(xx, alpha_true / 12 + beta_true * xx, color="black", linestyle="--", label=f"true: beta {beta_true}")
ax.axhline(0, color="grey", linewidth=0.6)
ax.axvline(0, color="grey", linewidth=0.6)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Stock vs market, five years of monthly excess returns")
ax.set_xlabel("market excess return (monthly)")
ax.set_ylabel("stock excess return (monthly)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(B_hat, bins=60, histtype="step", linewidth=1.5)
axes[0].axvline(beta_true, color="black", linewidth=0.8)
axes[0].set_title(f"Beta estimates (5 years monthly), SE {B_hat.std():.2f}")
axes[0].set_xlabel("estimated beta")
axes[1].hist(A_hat * 12, bins=60, histtype="step", linewidth=1.5, color="tab:orange")
axes[1].axvline(alpha_true, color="black", linewidth=0.8)
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].set_title(f"Alpha estimates (annual), SE {(A_hat * 12).std():.1%}")
axes[1].set_xlabel("estimated alpha per year")
for ax in axes:
    ax.set_ylabel("number of histories")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
months = np.arange(1, x_roll.size + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(months, true_path, color="black", linewidth=1.5, label="true beta")
ax.plot(months, roll36, color="tab:blue", label="rolling 36 months")
ax.plot(months, roll60, color="tab:orange", label="rolling 60 months")
ax.set_title("Rolling beta estimates, beta jumps from 0.8 to 1.3 in month 121")
ax.set_xlabel("month")
ax.set_ylabel("beta")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(ns[:, 0], bins=40, histtype="step", linewidth=1.5, label="daily, same day")
axes[0].hist(ns[:, 1], bins=40, histtype="step", linewidth=1.5, label="daily, Dimson")
axes[0].hist(ns[:, 2], bins=40, histtype="step", linewidth=1.5, label="monthly")
axes[0].axvline(beta_true, color="black", linewidth=0.8)
axes[0].set_title("Non-synchronous prices: beta estimates (true 1.2)")
axes[0].set_xlabel("estimated beta")
axes[0].legend(fontsize=8)

axes[1].scatter(b1, b2, s=6, alpha=0.5)
bb = np.linspace(b1.min(), b1.max(), 20)
axes[1].plot(bb, bb, color="grey", linestyle=":", label="45° line")
fit = stats.linregress(b1, b2)
axes[1].plot(bb, fit.intercept + fit.slope * bb, color="tab:red", label=f"fit, slope {fit.slope:.2f}")
axes[1].set_title("Betas of 500 stocks in two periods")
axes[1].set_xlabel("beta, period 1")
axes[1].set_ylabel("beta, period 2")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. A stock has volatility 25%, correlation 0.6 with the market, and the market has volatility 17%. Compute beta, $R^2$ and the
   idiosyncratic volatility.
2. For the stock of exercise 1, compute the standard error of beta from 3 years and from 10 years of monthly data.
3. A fund reports beta 1.3, $R^2$ 0.95 and alpha 1.5% per year with a t-statistic of 0.8 against its equity benchmark. **Interpret**
   these three numbers: what kind of fund is it, and what are you paying for?

### Intermediate

4. Show numerically that maximum likelihood gives the least-squares estimates: maximize the normal log-likelihood of the
   five-year sample over $(\alpha, \beta, \sigma_\varepsilon)$ with `scipy.optimize.minimize` and compare with `stats.linregress`.
5. Use the Dimson regression with lags 0, 1 and 2 when the stock reacts 50% today, 30% tomorrow and 20% the day after. How
   biased are the same-day and the one-lag estimates?

### Advanced

6. Show that $\operatorname{Var}(\hat\beta \mid x) = \sigma_\varepsilon^2/\sum_t (x_t - \bar x)^2$ by writing $\hat\beta = \beta + \sum_t c_t\varepsilon_t$ with $c_t = (x_t - \bar x)/\sum_s (x_s - \bar x)^2$. Check it
   by simulating many $\varepsilon$ for one fixed market path.
7. Vasicek shrinkage: replace the fixed $\tfrac23, \tfrac13$ with the weight $w = \operatorname{Var}(\beta)/(\operatorname{Var}(\beta) + \operatorname{SE}^2)$ toward the cross-sectional mean.
   For the 500 stocks, compare the mean squared error against the true betas of the raw, Blume-adjusted and Vasicek-adjusted
   period-1 estimates.

## 8. Solutions / Checks

In [ ]:
# 1.
b1_ex = 0.6 * 0.25 / 0.17
print(f"1. beta {b1_ex:.3f}, R^2 {0.36:.2f}, idiosyncratic vol {0.25 * np.sqrt(1 - 0.36):.1%}")

# 2.
for T in [3, 10]:
    print(f"2. {T:2d} years: SE(beta) = {0.25 * 0.8 / (0.17 * np.sqrt(12 * T)):.3f}")

# 3.
print(f"3. systematic vol share {0.95:.0%}; alpha SE = {1.5 / 0.8:.2f}% per year")

In [ ]:
# 4.
from scipy.optimize import minimize

def neg_loglik(theta):
    a, b, log_s = theta
    e = y - a - b * x
    s2 = np.exp(2 * log_s)
    return 0.5 * x.size * np.log(2 * np.pi * s2) + (e**2).sum() / (2 * s2)

mle = minimize(neg_loglik, x0=[0.0, 1.0, np.log(0.05)], method="Nelder-Mead",
               options={"xatol": 1e-10, "fatol": 1e-10, "maxiter": 20_000})
print(f"4. MLE alpha {mle.x[0]:.6f}, beta {mle.x[1]:.6f};  OLS alpha {lr.intercept:.6f}, beta {lr.slope:.6f}")

# 5.
def dimson_two_lags(rng, shares=(0.5, 0.3, 0.2), n_years=5):
    n = n_years * trading_days + 2
    xm = mkt_mu / trading_days + mkt_vol / np.sqrt(trading_days) * rng.standard_normal(n)
    ys = (beta_true * (shares[0] * xm[2:] + shares[1] * xm[1:-1] + shares[2] * xm[:-2])
          + idio_vol / np.sqrt(trading_days) * rng.standard_normal(n - 2))
    x0, x1, x2 = xm[2:], xm[1:-1], xm[:-2]
    one = np.ones_like(x0)
    c0 = np.linalg.lstsq(np.column_stack([one, x0]), ys, rcond=None)[0]
    c1 = np.linalg.lstsq(np.column_stack([one, x0, x1]), ys, rcond=None)[0]
    c2 = np.linalg.lstsq(np.column_stack([one, x0, x1, x2]), ys, rcond=None)[0]
    return c0[1], c1[1:].sum(), c2[1:].sum()

d5 = np.array([dimson_two_lags(rng) for _ in range(500)])
print(f"5. average beta: same day {d5[:, 0].mean():.3f}, lags 0-1 {d5[:, 1].mean():.3f}, lags 0-2 {d5[:, 2].mean():.3f}")

In [ ]:
# 6.
x_fixed = simulate_stock(60, rng)[0]
eps = idio_vol / np.sqrt(12) * rng.standard_normal((20_000, 60))
b6 = ols(np.broadcast_to(x_fixed, eps.shape), beta_true * x_fixed + eps)[1]
print(f"6. simulated SD {b6.std():.4f}, formula {idio_vol / np.sqrt(12) / np.sqrt(((x_fixed - x_fixed.mean())**2).sum()):.4f}")

# 7.
se_b1 = idio_vol / (mkt_vol * np.sqrt(60))
var_b_cs = max(b1.var() - se_b1**2, 0.0)
w_v = var_b_cs / (var_b_cs + se_b1**2)
vasicek = w_v * b1 + (1 - w_v) * b1.mean()
mse = lambda est: np.mean((est - true_betas) ** 2)
print(f"7. Vasicek weight {w_v:.2f};  MSE raw {mse(b1):.4f}, Blume {mse(2 / 3 * b1 + 1 / 3):.4f}, Vasicek {mse(vasicek):.4f}")

**1.** $\beta = 0.6 \cdot 25/17 = 0.88$, $R^2 = 0.36$, idiosyncratic volatility $25\%\sqrt{0.64} = 20\%$.

**2.** $\sigma_\varepsilon = 20\%$; $\operatorname{SE} = 20\%/(17\%\sqrt{36}) = 0.196$ over 3 years and $20\%/(17\%\sqrt{120}) = 0.107$ over 10 years.

**3.** 95% of the fund's variance is the benchmark, amplified 1.3 times: it behaves like a leveraged index fund. The alpha
of 1.5% has a standard error of about 1.9% (1.5/0.8) and is not distinguishable from zero. An investor could get most of
this exposure by holding 1.3 times the index (or the index plus futures) at much lower cost; the fee pays for leverage,
not for demonstrated skill.

**4.** The maximum-likelihood intercept and slope agree with least squares to the precision of the optimizer.

**5.** With the reaction spread over three days, the same-day beta captures only about half of the true beta (0.6), the
one-lag Dimson estimate about 80% (0.96), and the two-lag estimate recovers 1.2.

**6.** $\hat\beta - \beta = \sum_t c_t\varepsilon_t$ with $\sum_t c_t^2 = 1/\sum_t (x_t - \bar x)^2$, so the variance is $\sigma_\varepsilon^2/\sum_t(x_t-\bar x)^2$. The simulation
for a fixed market path matches.

**7.** The Vasicek weight is 0.78. Mean squared errors: raw 0.022, Blume 0.018, Vasicek 0.017. Both adjustments help;
Vasicek estimates the shrinkage weight from the data instead of fixing it at 2/3.

## 9. Conclusion

- The OLS beta is $\widehat{\operatorname{Cov}}(x,y)/\widehat{\operatorname{Var}}(x)$; with normal errors it is also the maximum-likelihood estimate.
- Variance splits into systematic $\beta^2\sigma_M^2$ and idiosyncratic $\sigma_\varepsilon^2$; $R^2 = \rho^2$ is the systematic share.
- Beta is precise (standard error shrinks with the number of observations); alpha is not (standard error $\sigma_\varepsilon/\sqrt T$).
- A high-beta fund beats a rising market without skill; the regression alpha removes that effect.
- Betas drift over time, daily betas of non-synchronous stocks are biased toward zero (Dimson fixes it), and extreme betas
  regress toward 1 (Blume and Vasicek adjustments).

Next: [07/02 Least squares and the normal equations](./02_least_squares_and_the_normal_equations.ipynb).